In [1]:
# =========================================================
# Setup — import libraries
# =========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv('telco_cleaned (1).csv')

print("Shape:", df.shape)
print(df.head())
print("\nChurn distribution:")
print(df['Churn'].value_counts())

Shape: (1000, 9)
  customerID  gender  tenure        Contract            PaymentMethod  \
0     TC0001    Male    44.0        One year  Credit card (automatic)   
1     TC0002    Male    59.0  Month-to-month         Electronic check   
2     TC0003  Female    46.0        Two year             Mailed check   
3     TC0004  Female    43.0        One year             Mailed check   
4     TC0005  Female    34.0  Month-to-month  Credit card (automatic)   

  InternetService  MonthlyCharges  TotalCharges Churn  
0     Fiber optic           57.26       2522.74    No  
1              No           71.35       4186.43   Yes  
2     Fiber optic           20.06        872.06    No  
3     Fiber optic           78.04       3292.39    No  
4              No           84.64       2875.34   Yes  

Churn distribution:
Churn
No     747
Yes    253
Name: count, dtype: int64


## Task 1 — Scaling Experiment with KNN
<!-- TODO: Train KNN three times with No Scaling, Min-Max, and StandardScaler. Compare accuracy. ✅ DONE -->

In [3]:
# =========================================================
# Task 1 — Scaling Experiment with KNN
# =========================================================

X = df[['tenure', 'MonthlyCharges', 'TotalCharges']]
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

knn = KNeighborsClassifier(n_neighbors=5)

# Run 1: No scaling
knn.fit(X_train, y_train)
acc_no_scale = accuracy_score(y_test, knn.predict(X_test))

# Run 2: MinMaxScaler
mm = MinMaxScaler()
knn.fit(mm.fit_transform(X_train), y_train)
acc_minmax = accuracy_score(y_test, knn.predict(mm.transform(X_test)))

# Run 3: StandardScaler
std = StandardScaler()
knn.fit(std.fit_transform(X_train), y_train)
acc_standard = accuracy_score(y_test, knn.predict(std.transform(X_test)))

print("Scaling Method      | Accuracy")
print("-" * 35)
print(f"No Scaling          | {acc_no_scale*100:.2f}%")
print(f"Min-Max             | {acc_minmax*100:.2f}%")
print(f"Standardization     | {acc_standard*100:.2f}%")

Scaling Method      | Accuracy
-----------------------------------
No Scaling          | 70.00%
Min-Max             | 70.00%
Standardization     | 70.00%


## Task 2 — Encoding Experiment
<!-- TODO: Encode categorical features with Label Encoding and One-Hot Encoding. Train KNN each time. Compare accuracy. ✅ DONE -->

In [4]:
# =========================================================
# Task 2 — Encoding Experiment
# =========================================================

# --- Label Encoding ---
le = LabelEncoder()
df['Contract_le'] = le.fit_transform(df['Contract'])
df['PaymentMethod_le'] = le.fit_transform(df['PaymentMethod'])
df['InternetService_le'] = le.fit_transform(df['InternetService'])

X_le = df[['tenure', 'MonthlyCharges', 'TotalCharges',
           'Contract_le', 'PaymentMethod_le', 'InternetService_le']]

X_train_le, X_test_le, y_train_le, y_test_le = train_test_split(
    X_le, y, test_size=0.2, random_state=42
)
std = StandardScaler()
knn.fit(std.fit_transform(X_train_le), y_train_le)
acc_label = accuracy_score(y_test_le, knn.predict(std.transform(X_test_le)))

# --- One-Hot Encoding ---
ohe = pd.get_dummies(df[['Contract', 'PaymentMethod', 'InternetService']],
                     drop_first=True)
X_ohe = pd.concat([df[['tenure', 'MonthlyCharges', 'TotalCharges']], ohe], axis=1)

X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(
    X_ohe, y, test_size=0.2, random_state=42
)
std2 = StandardScaler()
knn.fit(std2.fit_transform(X_train_ohe), y_train_ohe)
acc_ohe = accuracy_score(y_test_ohe, knn.predict(std2.transform(X_test_ohe)))

# Comparison table
print("Encoding Method     | Accuracy")
print("-" * 35)
print(f"Label Encoding      | {acc_label*100:.2f}%")
print(f"One-Hot Encoding    | {acc_ohe*100:.2f}%")

Encoding Method     | Accuracy
-----------------------------------
Label Encoding      | 73.00%
One-Hot Encoding    | 74.00%


## Written Analysis
<!-- TODO: Write 1 paragraph per experiment explaining why results differ. ✅ DONE -->

## Analysis

### Task 1 — Scaling Experiment
All three scaling methods (No Scaling, Min-Max, Standardization) produced
the same accuracy of 70.00% on this dataset. This is likely because the
dataset is relatively small and clean, so the difference in feature ranges
did not strongly affect KNN's distance calculations. In larger, more complex
datasets, scaling typically makes a significant difference because KNN relies
on distance — features with larger ranges (like TotalCharges) would otherwise
dominate the distance metric unfairly.

### Task 2 — Encoding Experiment
One-Hot Encoding (74.00%) slightly outperformed Label Encoding (73.00%).
This is expected because Label Encoding assigns arbitrary integers (0, 1, 2)
to categories like Contract types, which implies a false ordering — the model
treats "Two year" as mathematically greater than "Month-to-month", which is
misleading. One-Hot Encoding avoids this by creating separate binary columns
for each category, allowing KNN to treat them as equal and independent.